GSE116256: Acture Myeloid Leukemia

Source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE116256

In [ ]:
%pip install aeacus
%pip install -q scanpy scikit-learn

In [13]:
import os, tarfile, urllib.request
import numpy as np, pandas as pd, scanpy as sc, scipy.sparse as sp
from sklearn.metrics import accuracy_score

In [4]:
DATA_DIR = "deepdecon_aml"
TAR_PATH = "datasets.tar.gz"
URL      = "https://zenodo.org/records/7223362/files/datasets.tar.gz?download=1"

In [5]:
if not os.path.exists(DATA_DIR):
    urllib.request.urlretrieve(URL, TAR_PATH)
    with tarfile.open(TAR_PATH) as t:
        t.extractall(DATA_DIR)
    os.remove(TAR_PATH)

In [6]:
files = sorted(f"{DATA_DIR}/datasets/{n}" for n in os.listdir(f"{DATA_DIR}/datasets") if n.endswith(".txt"))

In [7]:
# creating ann data object
adatas = []
for f in files:
    sid = os.path.basename(f).replace("_norm_sc.txt", "")
    df  = pd.read_csv(f, sep=",", header=0, index_col=0)
    labels = df.iloc[:, 0].astype(str).str.strip()
    expr   = df.iloc[:, 1:].astype(np.float32)
    a = sc.AnnData(
        X   = sp.csr_matrix(expr.values),
        obs = pd.DataFrame({"subject": sid,
                            "y_true" : (labels == "malignant").astype(int).values,
                            "label"  : labels.values},
                           index=[f"{sid}_{bc}" for bc in df.index]),
        var = pd.DataFrame(index=expr.columns)
    )
    adatas.append(a)

In [8]:
adata = sc.concat(adatas, join="inner")
adata.var_names_make_unique()
print(f"{adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(adata.obs["label"].value_counts().to_string())

19,392 cells × 7,864 genes
label
malignant    12130
normal        7262


In [9]:
# data is already normalized
from aeacus import Profiler

profiler = Profiler(
    test_input = adata.copy(),
    norm_type  = False,
)
profiler.load()
result = profiler.profile()

/home/pandeyps/.pyenv/versions/3.11.9/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Model features: 3778
Missing features: 571 (15.11%)


In [10]:
adata.obs["score"] = result.obs.loc[adata.obs_names, "malignancy_score"].values
adata.obs["pred"]  = result.obs.loc[adata.obs_names, "malignancy_call"].values

In [15]:
y_true  = adata.obs["y_true"].values
y_pred  = (adata.obs["pred"] == "Malignant").astype(int).values

accuracy = accuracy_score(y_true, y_pred)
accuracy

0.44977310231023104